# Notebook 19: Self-Supervised Tabular & Time Series

## Why This Matters
Most tabular and time series data is unlabeled. The contrastive learning ideas from SimCLR and BERT apply directly — just with different augmentation strategies. SCARF uses feature corruption for tabular data. TS2Vec uses temporal cropping for time series. Both produce embeddings that support retrieval, clustering, and semi-supervised learning without any labels.

## Structure
1. From SimCLR to SCARF — the same objective, different augmentation (narrative)
2. SCARF: self-supervised contrastive learning for tabular data
3. TS2Vec: contrastive pretraining on time series
4. Evaluating without labels: clustering quality
5. Bridge to the capstone

## What You'll Learn
- How feature corruption augmentation is the tabular analog of image cropping
- How temporal cropping creates positive pairs for time series
- How to evaluate unsupervised representations with ARI and silhouette score


## Background

### The common pattern

All self-supervised representation learning follows the same template:

```
1. Define positive pairs: two views of the same sample that should be similar
2. Encode both views with (shared) encoder
3. Maximize similarity of positive pairs, minimize similarity of negative pairs
4. Downstream: freeze encoder, use representations for any task
```

The only thing that changes between methods is how "two views" are defined:

| Method | Domain | Positive pair definition |
|--------|--------|------------------------|
| SimCLR | Images | Two random augmentations of the same image |
| SBERT  | Text   | Paraphrase or NLI entailment pair |
| SCARF  | Tabular | Original row + feature-corrupted version |
| TS2Vec | Time series | Two overlapping temporal crops of the same series |

### SCARF augmentation

SCARF corrupts a tabular row by replacing a random subset of features with values drawn from the marginal distribution of that feature (not random noise — empirical marginal). This mimics the way image augmentation preserves semantic content while changing surface-level attributes.

The model must learn which features carry correlated signal — because corrupted features provide no useful information, the encoder is forced to rely on uncorrupted features that co-vary with the corrupted ones.


In [ ]:
# %pip install -q torch numpy scikit-learn matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42); np.random.seed(42)
device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. SCARF — Tabular Contrastive Learning

In [ ]:
class SCARFAugmentation:
    """SCARF augmentation: corrupt a random subset of features with marginal samples."""
    def __init__(self, data: np.ndarray, corruption_rate: float = 0.6):
        self.marginals = data  # store full dataset to sample from
        self.p = corruption_rate

    def __call__(self, x: np.ndarray) -> np.ndarray:
        """Corrupt each feature with probability p by replacing with a marginal sample."""
        x_corrupted = x.copy()
        mask = np.random.random(len(x)) < self.p
        if mask.any():
            n_corrupt = mask.sum()
            # Sample random rows from the dataset marginal distribution
            rand_rows = self.marginals[np.random.randint(0, len(self.marginals), n_corrupt)]
            x_corrupted[mask] = rand_rows[:, mask].diagonal() if rand_rows.ndim > 1                                  else rand_rows[mask]
            # Simpler: sample per-feature from the column marginal
            for j, corrupt in enumerate(mask):
                if corrupt:
                    x_corrupted[j] = self.marginals[np.random.randint(len(self.marginals)), j]
        return x_corrupted


class SCARFDataset(torch.utils.data.Dataset):
    def __init__(self, data: np.ndarray, corruption_rate: float = 0.6):
        self.data = data
        self.aug  = SCARFAugmentation(data, corruption_rate)

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        x = self.data[idx]
        return torch.tensor(x, dtype=torch.float32),                torch.tensor(self.aug(x), dtype=torch.float32)


class SCARFEncoder(nn.Module):
    def __init__(self, input_dim: int, embed_dim: int = 64, hidden: int = 128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),   nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Linear(hidden, embed_dim),
        )
        self.projector = nn.Sequential(
            nn.Linear(embed_dim, embed_dim), nn.ReLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def encode(self, x): return self.encoder(x)
    def forward(self, x): return self.projector(self.encoder(x))


# Generate a synthetic dataset with cluster structure
X, y_true = make_classification(n_samples=1000, n_features=20, n_informative=10,
                                 n_classes=5, n_clusters_per_class=1, random_state=42)
X = StandardScaler().fit_transform(X).astype(np.float32)

scarf_ds = SCARFDataset(X, corruption_rate=0.6)
loader   = DataLoader(scarf_ds, batch_size=64, shuffle=True)

encoder = SCARFEncoder(input_dim=20, embed_dim=32, hidden=64).to(device)
print(f"SCARF encoder params: {sum(p.numel() for p in encoder.parameters()):,}")

# NT-Xent loss (same as SimCLR)
def nt_xent(z1, z2, temperature=0.5):
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    z  = torch.cat([z1, z2])
    sim = torch.mm(z, z.T) / temperature
    mask = torch.eye(len(z), dtype=torch.bool, device=z.device)
    sim.masked_fill_(mask, -1e9)
    N = len(z1)
    targets = torch.cat([torch.arange(N, 2*N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)

optimizer = optim.Adam(encoder.parameters(), lr=3e-4)
print("Training SCARF (30 epochs)...")
for epoch in range(30):
    epoch_loss = 0
    for x_orig, x_corr in loader:
        z1 = encoder(x_orig.to(device))
        z2 = encoder(x_corr.to(device))
        loss = nt_xent(z1, z2)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1}: loss={epoch_loss/len(loader):.4f}")

In [ ]:
# Evaluate: do clusters in embedding space match true labels?
encoder.eval()
with torch.no_grad():
    embs = encoder.encode(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

# Cluster and compare to ground truth
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10).fit(embs)
ari  = adjusted_rand_score(y_true, kmeans.labels_)
sil  = silhouette_score(embs, kmeans.labels_, sample_size=500)

# Baseline: cluster raw features
kmeans_raw = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X)
ari_raw  = adjusted_rand_score(y_true, kmeans_raw.labels_)
sil_raw  = silhouette_score(X, kmeans_raw.labels_, sample_size=500)

print(f"{'':20s} {'ARI':>8} {'Silhouette':>12}")
print(f"Raw features:      {ari_raw:>8.3f} {sil_raw:>12.3f}")
print(f"SCARF embeddings:  {ari:>8.3f} {sil:>12.3f}")
print()
print("ARI > 0.3 suggests meaningful cluster structure in the embeddings.")
print("ARI ≈ 0   suggests no better than random (baseline).")

## 2. TS2Vec — Time Series Contrastive Learning

In [ ]:
class TS2VecEncoder(nn.Module):
    """Simplified TS2Vec encoder: dilated causal conv + instance/temporal contrast."""
    def __init__(self, input_dim: int, hidden: int = 64, out_dim: int = 32, depth: int = 4):
        super().__init__()
        layers = [nn.Conv1d(input_dim, hidden, kernel_size=3, padding=1)]
        for i in range(depth - 1):
            dilation = 2 ** i
            layers.append(nn.Conv1d(hidden, hidden, kernel_size=3,
                                    padding=dilation, dilation=dilation))
            layers.append(nn.GELU())
        layers.append(nn.Conv1d(hidden, out_dim, kernel_size=1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        # x: [B, T, C] → [B, C, T] for conv → [B, T, D]
        return self.net(x.permute(0, 2, 1)).permute(0, 2, 1)

    def encode(self, x):
        """Global representation: mean pool over time."""
        return self.forward(x).mean(dim=1)


def hierarchical_contrastive_loss(z1, z2, alpha=0.5, temporal_unit=0):
    """TS2Vec loss: instance-level + temporal-level contrast."""
    # Instance-level: each timestep of z1 should match z2
    z1_norm = F.normalize(z1, dim=-1)
    z2_norm = F.normalize(z2, dim=-1)

    # Instance contrast (across batch)
    B, T, D = z1.shape
    loss_instance = 0
    for t in range(T):
        loss_instance = loss_instance + nt_xent(z1_norm[:, t], z2_norm[:, t])
    loss_instance /= T

    # Temporal contrast (across time, within instance)
    z1_T = z1_norm.reshape(B * T, D)
    z2_T = z2_norm.reshape(B * T, D)
    loss_temporal = nt_xent(z1_T, z2_T)

    return alpha * loss_instance + (1 - alpha) * loss_temporal


# Generate synthetic multivariate time series
N_SERIES, T_LEN, N_CHANNELS = 500, 64, 5
# 3 distinct patterns
pattern_A = np.sin(np.linspace(0, 4*np.pi, T_LEN))
pattern_B = np.cos(np.linspace(0, 4*np.pi, T_LEN))
pattern_C = np.sign(np.sin(np.linspace(0, 6*np.pi, T_LEN)))

ts_data, ts_labels = [], []
for i in range(N_SERIES):
    if i < N_SERIES // 3:
        base = pattern_A
        ts_labels.append(0)
    elif i < 2 * N_SERIES // 3:
        base = pattern_B
        ts_labels.append(1)
    else:
        base = pattern_C
        ts_labels.append(2)
    ts = np.stack([base + np.random.randn(T_LEN) * 0.3 for _ in range(N_CHANNELS)], axis=1)
    ts_data.append(ts)

ts_data   = np.stack(ts_data).astype(np.float32)
ts_labels = np.array(ts_labels)
print(f"Time series dataset: {ts_data.shape}  (N × T × C)")


def temporal_crop_pair(x: torch.Tensor, min_len: int = 32):
    """Two random overlapping temporal crops of the same series."""
    T = x.shape[0]
    start1 = np.random.randint(0, T - min_len)
    start2 = np.random.randint(0, T - min_len)
    return x[start1:start1+min_len], x[start2:start2+min_len]


class TSDataset(torch.utils.data.Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        x = torch.tensor(self.data[idx])
        c1, c2 = temporal_crop_pair(x)
        return c1, c2


ts_loader = DataLoader(TSDataset(ts_data), batch_size=32, shuffle=True)
ts_encoder = TS2VecEncoder(input_dim=N_CHANNELS, hidden=32, out_dim=16).to(device)
ts_opt = optim.Adam(ts_encoder.parameters(), lr=3e-4)

print("Training TS2Vec (20 epochs)...")
for epoch in range(20):
    epoch_loss = 0
    for c1, c2 in ts_loader:
        z1 = ts_encoder(c1.to(device))
        z2 = ts_encoder(c2.to(device))
        # Simple instance-level loss for this demo
        B = z1.shape[0]
        z1m = z1.mean(dim=1); z2m = z2.mean(dim=1)
        loss = nt_xent(z1m, z2m)
        ts_opt.zero_grad(); loss.backward(); ts_opt.step()
        epoch_loss += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}: loss={epoch_loss/len(ts_loader):.4f}")

# Evaluate
ts_encoder.eval()
ts_X = torch.tensor(ts_data).to(device)
with torch.no_grad():
    ts_embs = ts_encoder.encode(ts_X).cpu().numpy()

km_ts  = KMeans(n_clusters=3, random_state=42, n_init=10).fit(ts_embs)
ari_ts = adjusted_rand_score(ts_labels, km_ts.labels_)
print(f"\nTS2Vec embedding ARI: {ari_ts:.3f}  (1.0 = perfect cluster recovery)")

## 3. Bridge to the Capstone

You've now built or used:
- Static embeddings (FastText) — Notebook 01
- Contextual embeddings (sentence-transformers) — Notebook 02
- Fine-tuning embeddings — Notebook 03
- Contrastive learning from scratch (SimCLR) — Notebook 04
- Momentum contrast (MoCo) — Notebook 05
- Self-supervised vision (DINO) — Notebook 06
- Multimodal embeddings (CLIP) — Notebook 07
- Probing and evaluation — Notebooks 08–09
- Dimensionality reduction and ANN — Notebooks 10–11
- Vector databases and RAG — Notebooks 12–13
- Drift detection — Notebook 14
- Cross-modal retrieval, fingerprinting, label noise — Notebooks 15–17
- Tabular and time series representations — Notebooks 18–19

The capstone assembles the drift detection and cross-modal pieces into a production-grade semantic drift monitor for image streams: the `drift-cam` prototype.


## Summary

| Method | Domain | Augmentation | Status |
|--------|--------|-------------|--------|
| SCARF | Tabular | Feature corruption from marginals | Active research |
| VIME | Tabular | Masking + value imputation | Active research |
| TS2Vec | Time series | Temporal cropping | Production use |
| TNC | Time series | Neighborhood sampling | Research |

**Next:** Notebook 20 — Capstone: Semantic Drift Monitor. End-to-end `drift-cam` prototype: CLIP embeddings + streaming anomaly detection on image streams.
